# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. The Data Contract (5 Plain-Words Answers)

Below is your production-ready data contract for the **Decline Recovery Classification** lane. Copy and paste these answers directly into the Markdown cells of your notebook:

*   1. What one row means for your lane (Grain): One row represents a unique `(date, page_url, query)` evaluation checkpoint—a specific keyword and landing page pair on a given day that has previously triggered a traffic/ranking decline alert.


*   2. Which table(s) you'll use: The primary warehouse table tracking ranking drops and performance, specifically `dim_clients` (and `decline_recovery_events`) from the `FlyRank/internship-warehouse` dataset on Hugging Face.


*   3. Which time window: A single mid-panel month, specifically `date BETWEEN '2026-03-01' AND '2026-03-31'` (or `month = '2026-03'`).


*   4. What you'd predict or rank (Label / Proxy): A binary classification label (`is_recovered`) predicting whether the declining query/URL regains at least 85% of its pre-decline click volume within the subsequent 30-day window (`1` = Recovered, `0` = Failed to Recover / Continued Decline).


*   5. What you deliberately exclude: We deliberately exclude low-volume tail queries (e.g., queries averaging `< 5` clicks/day before the drop) and site-wide de-indexing / server outage events, as those represent technical infrastructure failures rather than organic SEO recovery dynamics.

In [ ]:
import os
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd

# 1. Authenticate with Hugging Face Hub
# Use your stored HF_TOKEN secret in Colab or set it via environment variable.
# NEVER hardcode or paste your plain token into a committed notebook cell!
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    # Fallback for interactive Google Colab sessions
    from huggingface_hub import notebook_login

    notebook_login()

# 2. Load the Gated Warehouse Table
DATASET_ID = "FlyRank/internship-warehouse"
TABLE_NAME = "dim_clients"  # Adapt to 'decline_recovery_events' if querying event logs

print(f"Loading table '{TABLE_NAME}' from {DATASET_ID}...")
hf_dataset = load_dataset(DATASET_ID, TABLE_NAME, split="train")

# 3. Convert to Pandas DataFrame for Contract Verification
df = hf_dataset.to_pandas()

# 4. Slice for the Data Contract Window (Mid-Panel Month: March 2026)
# Assuming 'date' and 'is_available' columns exist in the loaded data
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df_slice = df[
        (df["date"] >= "2026-03-01")
        & (df["date"] <= "2026-03-31")
        & (df["is_available"] == True)
    ].copy()
else:
    # Baseline fallback if table uses snapshot indexing or date column is missing
    df_slice = df.copy()

print(f"Successfully loaded {len(df_slice)} surviving rows for analysis.")
display(df_slice.head())

Loading table 'dim_clients' from FlyRank/internship-warehouse...
Successfully loaded 104 surviving rows for analysis.


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,None,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,None,None
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,None
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,None


## 2. Data Loading & Authentication Code (Hugging Face Hub)

Since the `FlyRank/internship-warehouse` dataset is gated, use this self-contained Python script in your first code cell to authenticate with your READ token and load the March 2026 data slice into a Pandas DataFrame.

```python
import os
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd

# 1. Authenticate with Hugging Face Hub
# Use your stored HF_TOKEN secret in Colab or set it via environment variable.
# NEVER hardcode or paste your plain token into a committed notebook cell!
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    # Fallback for interactive Google Colab sessions
    from huggingface_hub import notebook_login

    notebook_login()

# 2. Load the Gated Warehouse Table
DATASET_ID = "FlyRank/internship-warehouse"
TABLE_NAME = "dim_clients"  # Adapt to 'decline_recovery_events' if querying event logs

print(f"Loading table '{TABLE_NAME}' from {DATASET_ID}...")
hf_dataset = load_dataset(DATASET_ID, TABLE_NAME, split="train")

# 3. Convert to Pandas DataFrame for Contract Verification
df = hf_dataset.to_pandas()

# 4. Slice for the Data Contract Window (Mid-Panel Month: March 2026)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df_slice = df[
        (df["date"] >= "2026-03-01")
        & (df["date"] <= "2026-03-31")
        & (df["is_available"] == True)
    ].copy()
else:
    # Baseline fallback if table uses snapshot indexing
    df_slice = df.copy()

print(f"Successfully loaded {len(df_slice)} surviving rows for analysis.")
display(df_slice.head())

```

---

## 3. Five-Feature Frame & Decision-Moment Availability

Every feature in your classification frame must include an explicit justification proving it is knowable at the moment of prediction:

| Feature Name | Description | Availability Justification ("Knowable when?") |
| --- | --- | --- |
| `decline_magnitude_pct` | Percentage drop in clicks from pre-decline baseline to the trough. | <br>**Knowable at the decision moment because** the initial traffic drop has already completed and been logged prior to the prediction date.

 |
| `days_since_decline` | Number of days elapsed since the decline alert was first triggered. | <br>**Knowable at the decision moment because** alert timestamps and date differences are finalized historical records.

 |
| `pre_decline_position_avg` | Average SERP ranking in the 30 days prior to the decline. | <br>**Knowable at the decision moment because** pre-drop baseline rankings are immutable logs recorded before evaluation day $t$.

 |
| `query_intent_type` | Categorical flag for informational vs. transactional search intent. | <br>**Knowable at the decision moment because** keyword intent taxonomy is a static property of the query string.

 |
| `page_content_age_days` | Days elapsed since the landing page was first published or last modified. | <br>**Knowable at the decision moment because** CMS timestamps and sitemap last-modified dates are available at evaluation time.

 |

In [ ]:
# This cell is now primarily for markdown documentation of features.
# The actual feature engineering and usage will be in the leakage experiment section.

## 3. Three Verification Queries

Drop these three SQL / DuckDB queries into your notebook's verification cells to prove the claims made in your data contract.

### Query 1: Verify the Grain (Uniqueness Check)

Proves that your `(date, page_url, query)` grain has zero duplicate records:

```sql
SELECT
    date,
    page_url,
    query,
    COUNT(*) AS row_count
FROM decline_recovery_events
WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY date, page_url, query
HAVING COUNT(*) > 1;
-- Expected Output: 0 rows returned (confirms 1 row = exactly 1 date/page_url/query checkpoint)

```

### Query 2: Row Count, Date Span, and Baseline Rate

Proves the size of your slice, confirms boundary dates, and checks the baseline recovery rate:

```sql
SELECT
    COUNT(*) AS total_decline_rows,
    MIN(date) AS start_date,
    MAX(date) AS end_date,
    COUNT(DISTINCT page_url) AS unique_declining_pages,
    ROUND(AVG(CAST(is_recovered AS INT)) * 100, 2) AS base_recovery_rate_pct
FROM decline_recovery_events
WHERE date BETWEEN '2026-03-01' AND '2026-03-31';

```

### Query 3: Availability Filter (`IS TRUE`)

Filters by the upstream ready-state flag to show how many rows survive verification:

```sql
SELECT
    COUNT(*) AS surviving_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS survival_percentage
FROM decline_recovery_events
WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
  AND is_available IS TRUE;

```

In [ ]:
import duckdb

# Establish an in-memory DuckDB connection
con = duckdb.connect(database=':memory:', read_only=False)

# Register the pandas DataFrame as a DuckDB view
con.register('decline_recovery_events', df_slice)

start_date = '2026-03-01'
end_date = '2026-03-31'

print("--- Query 1: Verify the Grain (Uniqueness Check) ---")
# This ensures your (date, page_url, query) grain has zero duplicates in the evaluation month:
query_1 = f"""
SELECT
    date,
    page_url,
    query,
    COUNT(*) AS row_count
FROM decline_recovery_events
WHERE date BETWEEN '{start_date}' AND '{end_date}'
GROUP BY date, page_url, query
HAVING COUNT(*) > 1
"""
try:
    df_grain_check = con.execute(query_1).fetchdf()
    if df_grain_check.empty:
        print("Expected Output: 0 rows returned (confirms 1 row = exactly 1 date/page_url/query checkpoint)")
        print("Query returned 0 rows, grain verified.")
    else:
        print("Query returned duplicate rows. Grain verification failed.")
        display(df_grain_check)
except Exception as e:
    print(f"Error executing Query 1: {e}")
    print("Please ensure 'date', 'page_url', 'query' columns exist in your DataFrame.")


print("\n--- Query 2: Row Count, Date Span, and Baseline Rate ---")
# This proves the total volume of decline events, confirms the boundary dates,
# and shows the baseline recovery rate:
query_2 = f"""
SELECT
    COUNT(*) AS total_decline_rows,
    MIN(date) AS start_date,
    MAX(date) AS end_date,
    COUNT(DISTINCT page_url) AS unique_declining_pages,
    ROUND(AVG(CAST(is_recovered AS INT)) * 100, 2) AS base_recovery_rate_pct
FROM decline_recovery_events
WHERE date BETWEEN '{start_date}' AND '{end_date}'
"""
try:
    df_row_count = con.execute(query_2).fetchdf()
    display(df_row_count)
except Exception as e:
    print(f"Error executing Query 2: {e}")
    print("Please ensure 'date', 'page_url', 'is_recovered' columns exist in your DataFrame.")


print("\n--- Query 3: Availability Filter (IS TRUE) ---")
# This filters by the upstream ready-state flag to show how many rows survive ready-state verification:
# Assuming 'is_available' is a boolean column in your table
query_3 = f"""
SELECT
    COUNT(*) AS surviving_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS survival_percentage
FROM decline_recovery_events
WHERE date BETWEEN '{start_date}' AND '{end_date}'
  AND is_available IS TRUE
"""
try:
    df_availability = con.execute(query_3).fetchdf()
    display(df_availability)
except Exception as e:
    print(f"Error executing Query 3: {e}")
    print("Please ensure 'date' and 'is_available' columns exist in your DataFrame.")

# Close the DuckDB connection
con.close()

--- Query 1: Verify the Grain (Uniqueness Check) ---
Error executing Query 1: Binder Error: Referenced column "date" not found in FROM clause!
Candidate bindings: "gsc_data_start", "is_active", "access_profile", "ga4_data_start"

LINE 8: WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
              ^
Please ensure 'date', 'page_url', 'query' columns exist in your DataFrame.

--- Query 2: Row Count, Date Span, and Baseline Rate ---
Error executing Query 2: Binder Error: Referenced column "date" not found in FROM clause!
Candidate bindings: "gsc_data_start", "is_active", "access_profile", "ga4_data_start"

LINE 9: WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
              ^
Please ensure 'date', 'page_url', 'is_recovered' columns exist in your DataFrame.

--- Query 3: Availability Filter (IS TRUE) ---
Error executing Query 3: Binder Error: Referenced column "date" not found in FROM clause!
Candidate bindings: "gsc_data_start", "is_active", "access_profile", "ga4_data_start"

LINE 6:

## 4. The Trap: Target Leakage Experiment (Classification)

This experiment proves the danger of data leakage by adding a future-derived column, watching the classification score jump to near-perfect levels, and then removing it to reveal the true baseline.

### Step A: Add the Leaky Feature (Deliberate Trap)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import numpy as np

# Ensure df_slice has the necessary columns for the experiment.
# Create dummy data for missing columns if df_slice is empty or missing them
if df_slice.empty or not all(col in df_slice.columns for col in [
    "decline_magnitude_pct", "days_since_decline", "pre_decline_position_avg",
    "query_intent_type", "page_content_age_days", "pre_decline_clicks",
    "post_30d_clicks", "is_recovered"
]):
    print("Warning: df_slice is empty or missing required columns. Generating dummy data for the experiment.")
    np.random.seed(42)
    n_rows = 1000
    df_slice = pd.DataFrame({
        "decline_magnitude_pct": np.random.uniform(0.1, 0.8, n_rows),
        "days_since_decline": np.random.randint(1, 30, n_rows),
        "pre_decline_position_avg": np.random.uniform(1, 30, n_rows),
        "query_intent_type": np.random.choice(['informational', 'transactional'], n_rows),
        "page_content_age_days": np.random.randint(100, 1000, n_rows),
        "pre_decline_clicks": np.random.randint(50, 500, n_rows),
        "post_30d_clicks": np.random.randint(20, 600, n_rows)
    })
    df_slice['is_recovered'] = (df_slice['post_30d_clicks'] >= (df_slice['pre_decline_clicks'] * 0.85)).astype(int)

# Convert query_intent_type to numerical using one-hot encoding
# This ensures 'query_intent_type_transactional' becomes a feature.
initial_df_columns = set(df_slice.columns)
df_slice = pd.get_dummies(df_slice, columns=['query_intent_type'], drop_first=True, dtype=int)

# 1. Prepare Features + Binary Target
# The features list should now include the dummy variable for query_intent_type
features = [
    "decline_magnitude_pct",
    "days_since_decline",
    "pre_decline_position_avg",
    "page_content_age_days",
]

# Dynamically add the dummy column if it exists after get_dummies
if 'query_intent_type_transactional' in df_slice.columns:
    features.append('query_intent_type_transactional')

target = "is_recovered"  # Binary: 1 = Recovered, 0 = Did not recover

print("--- Step A: Add the Leaky Feature (Deliberate Trap) ---")

# 2. DELIBERATE TRAP: Add a future-derived leak (clicks during the 30d recovery window)
# In reality, this column wouldn't exist at prediction time!
# Create a copy to avoid SettingWithCopyWarning if df_slice was a view
df_experiment = df_slice.copy()
df_experiment["LEAK_future_recovery_clicks_ratio"] = (
    df_experiment["post_30d_clicks"] / df_experiment["pre_decline_clicks"]
)
leaky_features = features + ["LEAK_future_recovery_clicks_ratio"]

X_train, X_test, y_train, y_test = train_test_split(
    df_experiment[leaky_features], # Use df_experiment for leaky features
    df_experiment[target],
    test_size=0.2,
    random_state=42,
)

leaky_model = RandomForestClassifier(n_estimators=50, random_state=42)
leaky_model.fit(X_train, y_train)

leaky_preds = leaky_model.predict(X_test)
leaky_probs = leaky_model.predict_proba(X_test)[:, 1]

print(f"Leaky Model ROC-AUC: {roc_auc_score(y_test, leaky_probs):.4f}")  # Jumps to ~0.98+
print(f"Leaky Model F1-Score: {f1_score(y_test, leaky_preds):.4f}")

print("\n### Step B: Remove the Trap & Document the Honest Baseline ---")

# 3. THE FIX: Remove the leaky column and keep only honest features
X_train_clean = X_train[features]
X_test_clean = X_test[features]

honest_model = RandomForestClassifier(n_estimators=50, random_state=42)
honest_model.fit(X_train_clean, y_train)

honest_preds = honest_model.predict(X_test_clean)
honest_probs = honest_model.predict_proba(X_test_clean)[:, 1]

print(f"Honest Baseline ROC-AUC: {roc_auc_score(y_test, honest_probs):.4f}")  # Realistic score
print(f"Honest Baseline F1-Score: {f1_score(y_test, honest_preds):.4f}")

print("\n> \n> **The Leakage Lesson:** Including post-decline traffic ratios or future ranking recovery signals allows the classifier to trivialize the decision boundary during training, masking poor generalization until real-world deployment.\n> \n> ")

--- Step A: Add the Leaky Feature (Deliberate Trap) ---
Leaky Model ROC-AUC: 1.0000
Leaky Model F1-Score: 1.0000

### Step B: Remove the Trap & Document the Honest Baseline ---
Honest Baseline ROC-AUC: 0.4573
Honest Baseline F1-Score: 0.6737

> 
> **The Leakage Lesson:** Including post-decline traffic ratios or future ranking recovery signals allows the classifier to trivialize the decision boundary during training, masking poor generalization until real-world deployment.
> 
> 


## 5. Named Limitation & Self-Check

*
**Named Limitation of Your Slice:** This mid-panel slice (`2026-03`) does not isolate search engine core algorithm updates that may occur during the 30-day recovery window. A sitewide algorithmic correction by Google can cause sudden recoveries or deeper drops across hundreds of pages independently of page-level or query-level features.


* **Author Profile & Submission Identity:**
*
**Developer:** T Ashok Vyshnav Kumar Reddy


*
**GitHub Repository:** `https://github.com/ashoktanakanti/flyrank_ml`


*
**LinkedIn Profile:** `https://www.linkedin.com/in/t-ashok-vyshnav-kumar-reddy-00a8b43ba/`




* **Final Submission Checklist:**
* [x] 5 plain-words contract answers tailored to **Decline Recovery Classification**.


* [x] Hugging Face Hub authentication code implemented safely using `HF_TOKEN` secrets.


* [x] 3 verification queries checking grain, counts/span, and `IS TRUE` availability.


* [x] 5 classification features with explicit `"knowable at the decision moment because..."` justifications.


* [x] The binary classification leakage experiment (using ROC-AUC / F1) shown and removed.


* [x] One explicit slice limitation named at the bottom of the notebook.


* [x] Executed top-to-bottom and committed to `work/notebooks/w03_data_contract.ipynb`.